# Wave 5 — three gross-code LPU operations, fail-fast

Short report on the three Wave-5 campaigns run on BB(12) = [[144,12,12]] (Tour-de-Gross LPU), all
complete on the local 24-core box:

| run | outdir | finished |
|---|---|---|
| **LPU idle** — bare LPU held for `rounds` QEC cycles | `runs/framework/bb144/lpu_idle` | 2026-07-24 12:57 |
| **Shift automorphism** — the 14-timestep swap circuit, `C` repeats, `delta = y` | `runs/framework/bb144/automorphism` | 2026-07-24 21:08 |
| **Joint Pauli** — `Ybar_1` measured through the whole LPU | `runs/framework/bb144/joint_pauli` | 2026-07-25 01:10 |

Common setup: `p_ref = 5e-3`, seed 42, `adaptive_shots_max = 3000`, full-DEM (`-XYZ`, non-CSS)
**Relay-BP with the tuned-cheap `num_sets=20`** — *not* the paper's 600. These are fail-fast
numbers: an upper bound on error, a lower bound on what the operation can do.

A fourth operation — the gross-to-gross **inter-module** gate — has **not** been run yet; see the
last section.

In [ ]:
# src/ is an editable install (`pip install -e .`) - modules import directly, no sys.path needed.
import pathlib
import numpy as np
import matplotlib.pyplot as plt

from lambda_analysis import (load_run, fill_spectrum, reweight_filled, rw_stats, eps_stats,
                             mass_window_p_max, zero_bin_fraction)

RUNS = pathlib.Path("../../runs/framework/bb144")
OPS = {"lpu_idle": "LPU idle", "automorphism": "shift automorphism", "joint_pauli": "Y1 joint Pauli"}
colors = {"lpu_idle": "#2c7fb8", "automorphism": "#d95f0e", "joint_pauli": "#756bb1"}

runs = {k: load_run(RUNS / k) for k in OPS}
filled = {k: fill_spectrum(r.spectrum) for k, r in runs.items()}
P_TARGET = 1e-3          # the campaign's reporting point
print({k: (len(r.spectrum.weights), r.cycles) for k, r in runs.items()})

## 1. Summary

`eps` is per-cycle under `cycles_of`: QEC rounds for idle (12), the repeated-measurement rounds
`lpu_C` for the automorphism and `Ybar_1` (10 each). `headroom` is the rule-of-three exposure from
sampled-but-empty bins — reweighted values are lower bounds, headroom is how far they could rise.

In [ ]:
hdr = f"{'op':<20}{'N_exp':>10}{'cyc':>5}{'D':>5}{'w0':>5}{'LER(1e-3)':>12}{'+-se':>10}{'head':>10}{'eps':>11}{'zero%':>7}"
print(hdr); print("-" * len(hdr))
for k, label in OPS.items():
    r, s = runs[k], filled[k]
    L, se, head = rw_stats(s, P_TARGET)
    eps = eps_stats(s, P_TARGET, r.cycles)[0]
    D = r.distance["distance"] if r.distance else None
    w0 = r.distance["onset"] if r.distance else None
    print(f"{label:<20}{r.spectrum.n_expanded:>10d}{r.cycles:>5d}"
          f"{(D if D else '-'):>5}{(w0 if w0 else '-'):>5}"
          f"{L:>12.2e}{se:>10.1e}{head:>10.1e}{eps:>11.2e}{100*zero_bin_fraction(s):>7.0f}")

print("\nTechnique-I ansatz (f5, all unpinned):")
for k, label in OPS.items():
    a = runs[k].ansatz["params"]
    print(f"  {label:<20} w0={a['w0']:>5.1f}  f0={a['f0']:.2e}  g1={a['gamma1']:>5.2f}  "
          f"g2={a['gamma2']:>5.2f}  wc={a['wc']:>6.1f}   (cost {runs[k].ansatz['cost']:.1f}, "
          f"{runs[k].ansatz['n_points']} pts)")

## 2. The importance-sampled failure spectrum

`f(w)` = fraction of weight-`w` fault configurations the decoder gets wrong — the raw measurement
everything else is built on. The reweighted LER of the next section is just `sum_w f(w) P(w|p)`,
so this is where a sick operation shows itself first.

Points are the sampled bins with binomial error bars; open markers on the floor are
zero-failure bins (plotted at the rule-of-three upper limit `3/T`, i.e. an upper bound, not a
measurement). Lines are the Technique-I `f5` fits. Dashed verticals mark the mean fault weight at
p = 1e-3, `N_exp * q_base * (p/p_ref)` — the weights that actually carry the mass there.

In [ ]:
from importance_sampling import failure_spectrum_ansatz

fig, ax = plt.subplots(figsize=(6.8, 4.6))
a_sat = 1 - 2.0 ** -12                      # K = 12 saturation, f(w) -> 1 - 2^-K

for k, label in OPS.items():
    r = runs[k]
    w = np.asarray(r.spectrum.weights, float)
    T = np.asarray(r.spectrum.trials, float)
    F = np.asarray(r.spectrum.failures, float)
    f = F / T
    se = np.sqrt(np.clip(f * (1 - f), 0, None) / T)
    hit, zero = F > 0, F == 0
    ax.errorbar(w[hit], f[hit], yerr=se[hit], fmt="o", ms=3.5, lw=.8, color=colors[k],
                label=f"{label}  ({int(hit.sum())}/{len(w)} bins with failures)")
    ax.plot(w[zero], 3.0 / T[zero], "v", ms=3.5, mfc="none", mew=.8, color=colors[k], alpha=.55)

    p = runs[k].ansatz["params"]
    wg = np.linspace(max(p["w0"], 1), w.max(), 400)
    ax.plot(wg, failure_spectrum_ansatz(wg, p["w0"], p["f0"], a_sat, model="f5",
                                        gamma1=p["gamma1"], gamma2=p["gamma2"], wc=p["wc"]),
            "-", lw=1.2, color=colors[k], alpha=.75)
    w_bar = r.spectrum.n_expanded * r.spectrum.q_base * (P_TARGET / r.p_ref)
    ax.axvline(w_bar, color=colors[k], ls="--", lw=.9, alpha=.5)

ax.axhline(a_sat, color="k", ls=":", lw=.8)
ax.text(900, a_sat * 1.1, r"$1-2^{-K}$", fontsize=7, va="bottom", ha="right")
ax.set(xscale="log", yscale="log", xlabel="fault weight $w$", ylabel="$f(w)$",
       ylim=(1e-4, 4), xlim=(1, 1300),
       title="Measured failure spectra (points) and $f5$ fits (lines)")
ax.legend(fontsize=7, loc="upper left"); ax.grid(alpha=.25, which="both")
fig.tight_layout()

## 3. Logical error rate

Points = importance-sampled spectrum, gap-filled, reweighted; lines = the Technique-I `f5` fit.
Curves are cut at `mass_window_p_max` (4 sigma of binomial mass inside the sampled window) —
beyond it reweighting is no longer unbiased.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.4))

for k, label in OPS.items():
    r, s = runs[k], filled[k]
    res = np.load(RUNS / k / "result.npz")
    pg = res["p_values"]
    ok = pg <= mass_window_p_max(s)
    ax.plot(pg[ok], reweight_filled(s, pg[ok]), "o", ms=4, color=colors[k], label=f"{label} (IS)")
    ax.plot(res["ansatz_p"], res["ansatz_P"], "-", lw=1.4, color=colors[k], alpha=.75,
            label=f"{label} (ansatz)")

ax.axvline(P_TARGET, color="k", ls=":", lw=.8)
ax.set(xscale="log", yscale="log", xlabel="physical error rate p", ylabel="logical error rate",
       ylim=(1e-12, 2), title="Wave-5 gross-code LPU operations (relay num_sets=20)")
ax.legend(fontsize=7, ncol=1, loc="lower right"); ax.grid(alpha=.25, which="both")
fig.tight_layout()

## 4. What the three runs say

**Cost tracks DEM size.** Expanded mechanism counts are 5.2e5 (idle) / 1.7e6 (automorphism) /
2.9e6 (`Ybar_1`), and the LER at p = 1e-3 orders the same way: ~2e-5, ~6e-3, ~0.5. `Ybar_1` is the
whole LPU in one operation, so at fixed p it carries roughly 6x the idle fault load; its shallow
`gamma1 = 4.55` is that, not a decoder pathology.

**Idle is the only run with Technique II**: D = 10 (bound), `w0` = 5, `f0*` = 4.8e-15, which meets
the paper's `d <= 10` bound for the LPU idle. Technique II was **dropped** from the automorphism
and `Ybar_1` configs — BP-OSD runs ~2 h/decode at 2e5+ columns (the smoke stalled at 6/50 trials
in 10 h). Their `result.npz` therefore carries `distance = 0`, `onset = nan` **by design**, and
their fits are `w0`/`f0`-free. The paper likewise has no `Ybar_1` row in Table 2.

**Both IS sweeps ended on the stop rule, not exhaustion** (3 consecutive zero-failure bins at
`shots_max = 3000`): the automorphism skipped 14 lower weights, `Ybar_1` skipped 1. Skipped
weights sit below the onset and contribute ~0.

### Two caveats before quoting any of this

1. **Never read `is_P_logical` straight out of `result.npz`.** The raw IS sum runs only over
   *sampled* weights, so a strided tail saturates at exactly `1/stride` instead of going to 1:
   idle is stride 1 (-> 1.0), the automorphism stride 4 (-> 0.250), `Ybar_1` stride 6 (-> 0.1667).
   It is low by ~stride at *every* p. `f(w)` itself is healthy (1.0 in the top bins). The cell
   below shows gap-filling closing the gap against the ansatz.
2. **Decoder handicap.** `num_sets = 20`, not the paper's 600. Combined with `p_ref = 5e-3` vs the
   paper's 1e-4 importance-sampling prior, these numbers are directionally comparable to Table 3,
   not numerically.

In [ ]:
# The stride artifact, and gap-filling as the fix.
print(f"{'op':<20}{'stride':>8}{'raw sat.':>10}{'1/stride':>10}   | at p=1e-3: {'raw':>10}{'filled':>10}{'ansatz':>10}")
for k, label in OPS.items():
    r, s = runs[k], filled[k]
    res = np.load(RUNS / k / "result.npz")
    w = np.asarray(r.spectrum.weights)
    stride = int(np.median(np.diff(w)))
    i = int(np.argmin(np.abs(res["p_values"] - P_TARGET)))
    print(f"{label:<20}{stride:>8d}{res['is_P_logical'][-1]:>10.4f}{1/stride:>10.4f}   |            "
          f"{res['is_P_logical'][i]:>10.2e}{reweight_filled(s, [P_TARGET])[0]:>10.2e}"
          f"{res['ansatz_P'][i]:>10.2e}")

## 5. Inter-module gate — NOT RUN YET

The fourth operation, and the one Wave 6 is built around: a **gross-to-gross adapter** joining two
[[144,12,12]] modules to measure `Xbar_1 (x) Xbar_1` across them. Branch `wave6-intermodule`.

**Built and green:**

* `AdapterGraph` descriptor + cross-module `U_B` derivation, `build_adapter_cycle`
  (bridge Bell checks), `build_joint_x1x1_circuit`;
* wired into `experiment_runner` and `lambda_analysis` (`cycles_of` already routes
  `inter_module` -> `lpu_C`), idle noise threaded through `build_lpu_cycle`;
* configs `gross_intermodule_{r1,r10}.yaml`; a commented bb288-class manifest block
  (48 cpus / 96 h / 64 G);
* the `U_B` gauge-check anchoring fix at `c = 0`, which took the circuit distance 2 -> 10.

**Open blocker.** The `obs0` readout floors at LER ~ 0.4 — a bridge Bell gauge issue, the same
*class* of bug as the `Ybar_1` framing bug (outcome bit not closed by detectors) that was fixed in
99db8351 by anchoring through the return boundary. Localisation is actively in progress
(floor-localisation matrix, idle-on/off discriminators, full-size `f(w)` spectrum). Trust Monte
Carlo here, **not** graphlike distance.

**Gate before this run joins the report** (promoted to the runbook after `Ybar_1`): a **weight-1
degeneracy scan showing zero same-syndrome / different-action groups**, plus a clean floor window,
tableau checks, and a green suite. Once that passes: `r1` for the floor, `r10` for the spectrum,
then Technique I only (Technique II is out of reach at this column count, as for the other two
LPU ops).

The cell below picks the run up automatically once it exists.

In [ ]:
im = RUNS / "inter_module"
if (im / "spectrum.json").exists():
    r = load_run(im); s = fill_spectrum(r.spectrum)
    L, se, head = rw_stats(s, P_TARGET)
    print(f"inter-module: N_exp={r.spectrum.n_expanded}  cycles={r.cycles}  "
          f"done={r.done_fraction:.0%}  LER(1e-3)={L:.2e} +-{se:.1e} (head {head:.1e})")
else:
    print(f"inter-module not run yet - no spectrum at {im}")
    print("blocked on the obs0 LER~0.4 floor (bridge Bell gauge); branch wave6-intermodule")

## 6. Next

* **Cluster re-run, staged but never submitted**: `experiments/slurm/submit_lpu.sh` fires all three
  as independent SLURM + podman jobs (16 / 16 / 32 cpus, 24 / 24 / 48 h), resumable, targeting the
  configs by path. That is what buys `Ybar_1` the paper's `num_sets = 600` relay and a real
  Technique II.
* **Decoder variant**: a `decoder_p` knob (`CalibratedRelayBP`, ported from the K=4 work) as a
  separate config, to separate decoder miscalibration from circuit cost in the `Ybar_1` number.
* **Contiguous low-weight tails** for the automorphism and `Ybar_1` if their low-p ends are ever
  quoted as point values rather than through the fit.